In [1]:
from mypy import *

In [2]:
x, y = fast_meshgrid(slm.resolution[0], slm.resolution[1], slm.pixel_size)
hg00 = laser(.77, 96, hermite_gauss(0, 0)).complex_amplitude(x, y, 0) * np.exp(-2j * pi * 0.0057870370 * y)
hg01 = laser(.77, 96, hermite_gauss(0, 1)).complex_amplitude(x, y, 0) * np.exp(2j * pi * 0.0057870370 * y)

In [3]:
hg = (hg00) + (hg01)

In [4]:
imwrite(array=None, save='C:/Users/Pepper/Desktop/1.bmp')

error: OpenCV(4.8.0) D:\a\opencv-python\opencv-python\opencv\modules\imgcodecs\src\loadsave.cpp:787: error: (-215:Assertion failed) !_img.empty() in function 'cv::imwrite'


## 超参数

In [ ]:
expo_time = r'1 s'
power = r'1 $\rm mW$'
title = r'{}: {} (曝光时间: ' + expo_time + r'; 激光功率: ' + power + ')'

In [ ]:
dir_spade = './spade/2/s2_{}.tif'

spade.point_1 = (128, 106)
spade.point_2 = (128, 400)

spade.characteristic_width = 96 / dmd.pixel_size

In [ ]:
dir_di = './di/2/s2_{}.tif'

di.zero_point = 44
di.pixel_size = 2 * qcmos.pixel_size / dmd.pixel_size

In [ ]:
samples = 100

In [ ]:
matplotlib_parameter.figsize(16, 4)

## SPADE

In [ ]:
spade_s = []
p1 = []
p2 = []

for i in range(samples):
    img = imread(dir_spade.format(i+1))
    spade_s.append(spade.estimator(img, mode='+-'))
    p1.append(spade.photon_number(img)[0])
    p2.append(spade.photon_number(img)[1])

spade_s = np.array(spade_s)
p1 = np.array(p1)
p2 = np.array(p2)

In [ ]:
plot((1, samples), spade_s,
     xlabel='单位: 帧', ylabel='估计值 (单位: DMD 像素)', title=title.format('SPADE', '估计值'), 
     save='spade_s.svg')

In [ ]:
plot((1, samples), p1,
     xlabel='单位: 帧', ylabel='光子数 (单位: 灰度值)', title=title.format('SPADE', 'P-光子数'), 
     save='spade_p-_n.svg')

In [ ]:
plot((1, samples), p2,
     xlabel='单位: 帧', ylabel='光子数 (单位: 灰度值)', title=title.format('SPADE', 'P+光子数'), 
     save='spade_p+_n.svg')

In [ ]:
print('SPADE: ')

spade_mean = spade_s.mean()
print('均值: ' + str(spade_mean))

spade_var = spade_s.var()
print('方差: ' + str(spade_var))

spade_n = (p1+p2).sum() / samples
print('平均光子数: ' + str(spade_n) + ' (单位: 灰度)')

spade_n_var = spade_var * spade_n
print('平均光子数*方差: ' + str(spade_n_var) + ' (单位: 灰度)')


qcmos.offset = 100
print('用公式换算后的光子数: ' + str(qcmos.grayscale_value_to_photon_number(spade_n)) + ' (单位: 个)')

In [ ]:
print('和 QCRB 的比值: ' + str(sqrt(spade_n_var) / spade.characteristic_width))

## DI

In [ ]:
di_s = []
n = []
for i in range(samples):
    img = imread(dir_di.format(i+1))
    di_s.append(di.estimator(img))
    n.append(di.photon_number(img))

di_s = np.array(di_s)
n = np.array(n)

In [ ]:
plot((1, samples), di_s, 
     xlabel='单位: 帧', ylabel='估计值 (单位: DMD 像素)', title=title.format('DI', '估计值'), 
     save='di_s.svg')

In [ ]:
plot((1, samples), n, 
     xlabel='单位: 帧', ylabel='光子数 (单位: 灰度值)', title=title.format('DI', '单张图片的总光子数'), 
     save='di_n.svg')

In [ ]:
print('DI: ')

di_mean = di_s.mean()
print('均值: ' + str(di_mean))

di_var = di_s.var()
print('方差: ' + str(di_var))

di_n = n.sum() / samples
print('平均光子数: ' + str(di_n))

di_n_var = di_var * di_n
print('平均光子数*方差: ' + str(di_n_var))

## 数据汇总

In [ ]:
from IPython.display import Math

def x(num1, num2):
    return '{' + str(num1) + r'\times' + str(num2) + '}'

tex_var = r'\frac{{\rm Var}[s_{_{\rm SPADE}}]}{{{\rm Var}[s_{_{\rm DI}}]}}=\frac'
tex_nvar = r'\frac{N_{_{\rm SPADE}}\times{\rm Var}[s_{_{\rm SPADE}}]}{{N_{_{\rm DI}}\times{\rm Var}[s_{_{\rm DI}}]}}=\frac'

tex_var = tex_var + '{' + str(spade_var) + '}' + '{' + str(di_var) + '}' + '=' + str(spade_var / di_var)
tex_nvar = tex_nvar + x(spade_n, spade_var) + x(di_n, di_var) + '=' + str(spade_n_var / di_n_var)

In [ ]:
Math(tex_var)

In [ ]:
Math(tex_nvar)

## 自动生成实验报告模版

In [ ]:
import time, os

m, d = time.localtime()[1], time.localtime()[2]

if not os.path.exists('./{}.{}实验报告.md/'.format(m, d)):
    f = open('./{}.{}实验报告.md'.format(m, d), 'w')
    f.write('# {}.{} 实验报告\n'.format(m, d))
    f.write('## 前言\n')
    f.write('**激光功率: {}**\n'.format(power))
    f.write('**曝光时间: {}**\n'.format(expo_time))
    f.write('## 实验结果\n')
    f.write('**方差比值:**\n')
    f.write('\n $$ \n' + tex_var + '\n $$ \n')
    f.write('\n $$ \n' + tex_nvar + '\n $$ \n')
    f.write('**数据图:**\n')
    for line in ['![spade_s](./spade_s.svg)', '![spade_p-_n](./spade_p-_n.svg)', '![spade_p+_n](./spade_p+_n.svg)', 
                '![di_s](./di_s.svg)', '![di_n](./di_n.svg)']:
        f.write(line + '\n')
    f.close()